# Práctica LLMs y gramáticas forzadas.
## Bioinformática y medicina.
-   Laura Cabaleiro Pintos
-   Marcelo Ferreiro Sánchez


Descarga del dataset

In [1]:

!pip install pandas fastparquet tqdm


!pip uninstall llama-cpp-python -y

!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121 \
    --no-cache-dir


!pip install datasets


!pip install ipywidgets

!pip install torch

Found existing installation: llama_cpp_python 0.3.19
Uninstalling llama_cpp_python-0.3.19:
  Successfully uninstalled llama_cpp_python-0.3.19
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 254.6 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

dataset = load_dataset("IIC/ClinText-SP")
dataset["train"].to_parquet("data/train.parquet")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Creating parquet from Arrow format:   0%|          | 0/36 [00:00<?, ?ba/s]

105400009

In [3]:
import pandas as pd

df = pd.read_parquet("data/train.parquet")
print(df.columns)
print(df.head(3))

# Filtrar por fuente, por ejemplo wikidisease
filtered = df[df["source"] == "wikidisease"]
print(filtered.head())



# Número total de filas
print("Número de filas:", len(df))


print("Columnas:", df.columns)

Index(['text', 'source'], dtype='object')
                                                text       source
0  **Enfermedad**: Hiperparatiroidismo\n**Abstrac...  wikidisease
1  **Enfermedad**: Alveolitis alérgica extrínseca...  wikidisease
2  **Enfermedad**: Hipertensión arterial\n**Abstr...  wikidisease
                                                text       source
0  **Enfermedad**: Hiperparatiroidismo\n**Abstrac...  wikidisease
1  **Enfermedad**: Alveolitis alérgica extrínseca...  wikidisease
2  **Enfermedad**: Hipertensión arterial\n**Abstr...  wikidisease
3  **Enfermedad**: Hipertrigliceridemia\n**Abstra...  wikidisease
4  **Enfermedad**: Hipocalcemia\n**Abstract**\nLa...  wikidisease
Número de filas: 35996
Columnas: Index(['text', 'source'], dtype='object')


El objetivo diseñado para esta práctica sería extraer de cada fila del dataset la enfermedad del informe, el tipo de tratamiento que hay, sus probabilidades de éxito, los métodos de diagnostico y notas de diagnóstico también.

Por ejemplo para este informe:

**Enfermedad**: Cáncer de la ampolla de Vater
**Abstract**
El cáncer ampular, cáncer de la ampollla de Vater o carcinoma del ámpula de Vater, es una neoplasia poco frecuente, de la estructura compuesta por la confluencia del conducto colédoco y el conducto pancreático, que desembocan en el lumen del duodeno a través de una pequeña protuberancia en la mucosa que recibe el nombre de papila duodenal mayor.
**Tratamiento**
En la mayoría de los pacientes con cáncer ampular (probabilidad por encima del 50%) la cirugía curativa es exitosa en comparación con los pacientes con adenocarcinoma de páncreas (probabilidad por debajo del 10%).
Se proponen tres opciones para tratar esta patología, las cuales son:
Pancreaticoduodenectomía (procedimiento de Whipple)
Ampulectomía quirúrgica (escisión local quirúrgica)
Ampulectomía endoscópica
**Diagnóstico**
El diagnóstico de las neoplasias del área "periampular" es difícil, porque en el "área del ámpula", concurren las patologías pancreáticas, del tercio distal del conducto biliar común, el conducto pancreático y la mucosa duodenal adyacente.
El 95% de las lesiones del ámpula de Vater diagnosticadas por endoscopía son adenomas o adenocarcinomas.
Existen diversos métodos diagnósticos, cada uno con sus ventajas y desventajas:
Tomografía computarizada de abdomen con/sin medio de contrasteVentajas: Es un procedimiento rápido que facilita la observación de anormalidades vasculares e incluso puede observar la existencia de metástasis a nivel hepático.
Desventajas: Posee pobre sensibilidad para el diagnóstico de lesiones "periampulares". Implica la exposición a radiación ionizante. El medio de contraste utilizado puede generar nefrotoxicidad por lo cual tiene poca utilidad en aquellos pacientes con insuficiencia renal.Resonancia magnética con medio de contrasteVentajas: Detecta pequeñas lesiones hepáticas, el medio de contraste utilizado no causa lesión renal y es un método que no utiliza radiación ionizante.
Desventajas: Posee pobre sensibilidad para el diagnóstico de lesiones "periampulares". Es un procedimiento caro y prolongado.Colangiopancreatografía retrógada endoscópica (CPRE)Ventajas: Sirve como método diagnóstico y como medida terapéutica.
Desventajas: Se considera un método invasivo e incrementa el riesgo de infección postquirúrgica.Ultrasonido endoscópicoVentaja: Es útil para delimitar anatómicamente la lesión.
Desventajas: Se considera un método invasivo y puede provocar una pancreatitis luego del procedimiento.

Construir un JSON estilo:




In [4]:
{
  "disease": "Cáncer de la ampolla de Vater",
  "treatment_types": ["quirúrgico", "endoscópico"],
  "treatment_options": [
    "Pancreaticoduodenectomía (Whipple)",
    "Ampulectomía quirúrgica",
    "Ampulectomía endoscópica"
  ],
  "success_probability": "cirugía curativa: >50%",
  "diagnostic_methods": [
    "Tomografía computarizada abdomen",
    "Resonancia magnética",
    "Colangiopancreatografía retrógada endoscópica (CPRE)",
    "Ultrasonido endoscópico"
  ],
  "diagnostic_notes": "Área periampular difícil, riesgo de invasión de estructuras adyacentes, sensibilidad limitada de algunos métodos"
}

{'disease': 'Cáncer de la ampolla de Vater',
 'treatment_types': ['quirúrgico', 'endoscópico'],
 'treatment_options': ['Pancreaticoduodenectomía (Whipple)',
  'Ampulectomía quirúrgica',
  'Ampulectomía endoscópica'],
 'success_probability': 'cirugía curativa: >50%',
 'diagnostic_methods': ['Tomografía computarizada abdomen',
  'Resonancia magnética',
  'Colangiopancreatografía retrógada endoscópica (CPRE)',
  'Ultrasonido endoscópico'],
 'diagnostic_notes': 'Área periampular difícil, riesgo de invasión de estructuras adyacentes, sensibilidad limitada de algunos métodos'}

Usaremos un modelo un poco más grande que el del ejemplo, ya que encontramos que ese último tendía a generar JSONs no válidos debido a que no sabía terminarlos del modo correcto. Este tiene 3.5B parámetros y da error muchas menos veces. 

In [18]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="bartowski/Phi-3.5-mini-instruct-GGUF",
    filename="Phi-3.5-mini-instruct-Q4_K_M.gguf",
    n_ctx=2048,
    n_gpu_layers=-1,
    verbose=False,
)

llama_context: n_ctx_seq (2048) < n_ctx_train (131072) -- the full capacity of the model will not be utilized


Definimos la gramática, se ponen 3 rangos de probabilidad generales y el valor desconocido para evitar alucinaciones.

In [12]:
from llama_cpp import LlamaGrammar

grammar_text = r"""
root ::= "{" ws disease "," ws treatment-types "," ws treatment-options "," ws success "," ws diag-methods "," ws diag-notes "}"

disease           ::= "\"disease\": " string-val
treatment-types   ::= "\"treatment_types\": " arr
treatment-options ::= "\"treatment_options\": " arr
success           ::= "\"success_probability\": " probability
diag-methods      ::= "\"diagnostic_methods\": " arr
diag-notes        ::= "\"diagnostic_notes\": " string-val

probability ::= "\"alta\"" | "\"media\"" | "\"baja\"" | "\"desconocida\""

arr        ::= "[" ws (string-val ("," ws string-val)*)? ws "]"
string-val ::= "\"" char* "\""
char       ::= [^"\\\x00-\x1F] | "\\" escape
escape     ::= ["\\bfnrt]
ws         ::= [ \t\n\r]*
"""

grammar = LlamaGrammar.from_string(grammar_text)

Construcción de un JSON con 100 ejemplos del dataset

In [19]:
import json

N = 100
results = []

for i, row in df.head(N).iterrows():
    abstract_text = row['text']

    prompt = f"""Eres un extractor de información médica.
Dado el siguiente texto clínico en español, extrae SOLO la información que aparece explícitamente.
No inventes datos. Si no aparece información para un campo, deja la lista vacía o pon "desconocida".

TEXTO:
{abstract_text}

Responde únicamente con el JSON, sin explicaciones."""

    try:
        response = llm(prompt, grammar=grammar, max_tokens=400)
        generated_json_str = response["choices"][0]["text"]
        parsed_json = json.loads(generated_json_str)
        parsed_json["source_text"] = abstract_text[:100]  # guarda referencia
        results.append(parsed_json)
        print(f"[{i+1}/{N}] OK — {parsed_json.get('disease', '???')}")
    except json.JSONDecodeError as e:
        print(f"[{i+1}/{N}] ERROR JSON — {e}")
        results.append({"error": str(e), "raw": generated_json_str})
    except Exception as e:
        print(f"[{i+1}/{N}] ERROR — {e}")

# Guardar a archivo
output_path = "resultados_clinicos.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"\nGuardado en {output_path} — {len(results)} entradas")

[1/100] OK — Hiperparatiroidismo
[2/100] OK — Alveolitis alérgica extrínseca
[3/100] OK — Hipertensión arterial
[4/100] OK — Hipertrigliceridemia
[5/100] OK — Hipocalcemia
[6/100] OK — Hipocondría
[7/100] OK — Hipotiroidismo
[8/100] OK — Síndrome de Bertolotti
[9/100] OK — Cáncer de la ampolla de Vater
[10/100] ERROR JSON — Unterminated string starting at: line 19 column 23 (char 883)
[11/100] ERROR JSON — Unterminated string starting at: line 1 column 223 (char 222)
[12/100] OK — Enfermedad de La Peyronie
[13/100] OK — Estenosis de la arteria renal
[14/100] OK — Virus sincitial respiratorio
[15/100] OK — Síndrome de las piernas inquietas
[16/100] OK — Fibrosis retroperitoneal
[17/100] OK — Retroversión uterina
[18/100] OK — Síndrome de Reye
[19/100] OK — Fiebre reumática
[20/100] ERROR JSON — Unterminated string starting at: line 1 column 573 (char 572)
[21/100] OK — Fractura costal
[22/100] OK — Raquitismo
[23/100] OK — Síndrome de Char
[24/100] OK — Síndrome de Cushing
[25/100] OK —